# 02.3 - Sorting Algorithms

## Why learn sorting if Python has `sorted()`?
1. Interviewers ask you to **implement** merge sort / quicksort and explain them.
2. Their ideas - **divide & conquer**, **partitioning**, **counting** - solve many other problems.
3. Sorting is often the **first step** of a solution (intervals, two pointers, greedy).

## Where is sorting used?
| Real system | Use |
|---|---|
| Databases | `ORDER BY`, sort-merge joins, building indexes |
| E-commerce / search | Sort by price, rating, relevance |
| Leaderboards, rankings, feeds | Sort by score / time |
| Data science (pandas `sort_values`) | Analysis, percentiles, medians |
| Deduplication & grouping | Sort, then equal items are adjacent |
| Big-data (MapReduce, Spark) | Shuffle-and-sort phase, external merge sort |

## Summary table
| Algorithm | Best | Average | Worst | Space | Stable? | Notes |
|---|---|---|---|---|---|---|
| Bubble sort | O(n) | O(n²) | O(n²) | O(1) | | Teaching only |
| Selection sort | O(n²) | O(n²) | O(n²) | O(1) | | Minimum swaps |
| Insertion sort | O(n) | O(n²) | O(n²) | O(1) | | Great for small / nearly sorted |
| **Merge sort** | O(n log n) | O(n log n) | O(n log n) | O(n) | | Predictable, linked lists, external sort |
| **Quicksort** | O(n log n) | O(n log n) | O(n²) | O(log n) | | Fastest in practice (random pivot) |
| Heap sort | O(n log n) | O(n log n) | O(n log n) | O(1) | | See 01.6 |
| Counting sort | O(n + k) | O(n + k) | O(n + k) | O(k) | | Small integer range k |
| **Timsort (Python)** | O(n) | O(n log n) | O(n log n) | O(n) | | Merge + insertion sort hybrid |

**Stable** = equal elements keep their original order (matters when sorting by several keys).

## 1. The O(n²) sorts

In [1]:
def bubble_sort(a):
    a = a[:]
    n = len(a)
    for i in range(n):
        swapped = False
        for j in range(n - 1 - i):            # largest "bubbles" to the end
            if a[j] > a[j + 1]:
                a[j], a[j + 1] = a[j + 1], a[j]
                swapped = True
        if not swapped:                       # already sorted -> O(n) best case
            break
    return a

def selection_sort(a):
    a = a[:]
    for i in range(len(a)):
        min_i = min(range(i, len(a)), key=a.__getitem__)   # pick the smallest remaining
        a[i], a[min_i] = a[min_i], a[i]
    return a

def insertion_sort(a):
    a = a[:]
    for i in range(1, len(a)):
        key = a[i]
        j = i - 1
        while j >= 0 and a[j] > key:          # shift bigger elements right
            a[j + 1] = a[j]
            j -= 1
        a[j + 1] = key                        # like sorting playing cards in your hand
    return a

data = [5, 2, 9, 1, 5, 6, 0, -3]
for f in (bubble_sort, selection_sort, insertion_sort):
    print(f"{f.__name__:15}", f(data))
    assert f(data) == sorted(data)

bubble_sort     [-3, 0, 1, 2, 5, 5, 6, 9]
selection_sort  [-3, 0, 1, 2, 5, 5, 6, 9]
insertion_sort  [-3, 0, 1, 2, 5, 5, 6, 9]


## 2. Merge sort - divide & conquer
1. **Divide** the array in half. 2. **Sort** each half recursively. 3. **Merge** the two sorted halves.
```
[5 2 9 1]  ->  [5 2] [9 1]  ->  [5][2] [9][1]  ->  [2 5] [1 9]  ->  [1 2 5 9]
```
log n levels × O(n) merge work per level = **O(n log n)**, always.

In [2]:
def merge_sort(a):
    if len(a) <= 1:
        return a
    mid = len(a) // 2
    left = merge_sort(a[:mid])
    right = merge_sort(a[mid:])
    return merge(left, right)

def merge(left, right):
    result, i, j = [], 0, 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:              # <= keeps it stable
            result.append(left[i]); i += 1
        else:
            result.append(right[j]); j += 1
    result.extend(left[i:])
    result.extend(right[j:])
    return result

print(merge_sort(data))
assert merge_sort(data) == sorted(data)

[-3, 0, 1, 2, 5, 5, 6, 9]


## 3. Quicksort - partitioning
1. Pick a **pivot**. 2. **Partition**: smaller elements left, bigger right. 3. Recursively sort both sides.
A **random pivot** makes the O(n²) worst case (already-sorted input with a bad pivot) extremely unlikely.

In [3]:
import random

def quick_sort(a):                         # simple, readable version (extra space)
    if len(a) <= 1:
        return a
    pivot = random.choice(a)
    less = [x for x in a if x < pivot]
    equal = [x for x in a if x == pivot]
    greater = [x for x in a if x > pivot]
    return quick_sort(less) + equal + quick_sort(greater)

def quick_sort_inplace(a, lo=0, hi=None):  # interview version: Lomuto partition, O(log n) space
    if hi is None:
        hi = len(a) - 1
    if lo >= hi:
        return a
    p = random.randint(lo, hi)
    a[p], a[hi] = a[hi], a[p]              # move random pivot to the end
    pivot, store = a[hi], lo
    for i in range(lo, hi):
        if a[i] < pivot:
            a[i], a[store] = a[store], a[i]
            store += 1
    a[store], a[hi] = a[hi], a[store]      # pivot into its final place
    quick_sort_inplace(a, lo, store - 1)
    quick_sort_inplace(a, store + 1, hi)
    return a

print(quick_sort(data), quick_sort_inplace(data[:]))
assert quick_sort(data) == quick_sort_inplace(data[:]) == sorted(data)

[-3, 0, 1, 2, 5, 5, 6, 9] [-3, 0, 1, 2, 5, 5, 6, 9]


## 4. Counting sort - beating O(n log n) when values are small integers

In [4]:
def counting_sort(a, max_value):
    count = [0] * (max_value + 1)
    for x in a:
        count[x] += 1
    result = []
    for value, c in enumerate(count):
        result.extend([value] * c)
    return result

ages = [25, 31, 18, 25, 42, 18, 60]
print(counting_sort(ages, 120))     # O(n + k): perfect for ages, grades, ratings 1-5

[18, 18, 25, 25, 31, 42, 60]


## 5. Race them

In [5]:
import time
random.seed(0)
big = [random.randint(0, 10**6) for _ in range(3000)]

for f in (bubble_sort, insertion_sort, merge_sort, quick_sort, sorted):
    start = time.perf_counter()
    f(big)
    print(f"{f.__name__:15} {(time.perf_counter() - start) * 1000:8.1f} ms")

bubble_sort        508.1 ms
insertion_sort     179.0 ms
merge_sort           8.3 ms
quick_sort           9.1 ms
sorted               0.4 ms


## 6. Sorting in Python like a pro

In [6]:
students = [("amina", 90, 20), ("eric", 85, 22), ("grace", 90, 19), ("jean", 70, 22)]

print(sorted(students, key=lambda s: s[1], reverse=True))         # by score desc
print(sorted(students, key=lambda s: (-s[1], s[0])))              # score desc, then name asc
from operator import itemgetter
print(sorted(students, key=itemgetter(2, 0)))                     # by age, then name

# Stability lets you sort by several keys in passes (secondary key first)
by_name = sorted(students, key=itemgetter(0))
print(sorted(by_name, key=itemgetter(1)))                          # equal scores stay name-ordered

# Custom comparison (rarely needed)
from functools import cmp_to_key
print(sorted([3, 30, 34, 5, 9], key=cmp_to_key(lambda a, b: -1 if str(a)+str(b) > str(b)+str(a) else 1)))

[('amina', 90, 20), ('grace', 90, 19), ('eric', 85, 22), ('jean', 70, 22)]
[('amina', 90, 20), ('grace', 90, 19), ('eric', 85, 22), ('jean', 70, 22)]
[('grace', 90, 19), ('amina', 90, 20), ('eric', 85, 22), ('jean', 70, 22)]
[('jean', 70, 22), ('eric', 85, 22), ('amina', 90, 20), ('grace', 90, 19)]
[9, 5, 34, 3, 30]


---
## Solved problems

### 1. Squares of a Sorted Array - LC #977 (Easy)
Return squares in sorted order in O(n).

**Idea:** Biggest squares are at the two ends; merge from the outside in (two pointers).

In [7]:
def sorted_squares(nums):
    n = len(nums)
    result = [0] * n
    l, r = 0, n - 1
    for pos in range(n - 1, -1, -1):          # fill from the biggest
        if abs(nums[l]) > abs(nums[r]):
            result[pos] = nums[l] ** 2; l += 1
        else:
            result[pos] = nums[r] ** 2; r -= 1
    return result

assert sorted_squares([-4, -1, 0, 3, 10]) == [0, 1, 9, 16, 100]
assert sorted_squares([-7, -3, 2, 3, 11]) == [4, 9, 9, 49, 121]
print("Sorted Squares ")

Sorted Squares 


### 2. Sort Colors - LC #75 (Medium) Dutch National Flag
Sort an array of 0s, 1s and 2s in one pass, in place.

**Idea:** Three pointers: everything before `low` is 0, after `high` is 2, between is 1 (quicksort's 3-way partition).

In [8]:
def sort_colors(nums):
    low, mid, high = 0, 0, len(nums) - 1
    while mid <= high:
        if nums[mid] == 0:
            nums[low], nums[mid] = nums[mid], nums[low]
            low += 1; mid += 1
        elif nums[mid] == 1:
            mid += 1
        else:
            nums[mid], nums[high] = nums[high], nums[mid]
            high -= 1                     # don't move mid: the swapped-in value is unchecked

a = [2, 0, 2, 1, 1, 0]; sort_colors(a); assert a == [0, 0, 1, 1, 2, 2]
b = [2, 0, 1]; sort_colors(b); assert b == [0, 1, 2]
print("Sort Colors ")

Sort Colors 


### 3. Sort an Array - LC #912 (Medium)
Sort without built-ins in O(n log n).

In [9]:
def sort_array(nums):
    return merge_sort(nums)       # merge sort from above: guaranteed O(n log n)

assert sort_array([5, 2, 3, 1]) == [1, 2, 3, 5]
assert sort_array([5, 1, 1, 2, 0, 0]) == [0, 0, 1, 1, 2, 5]
print("Sort an Array ")

Sort an Array 


### 4. Largest Number - LC #179 (Medium)
Arrange numbers to form the largest number (as a string).

**Idea:** Custom comparator: `a` goes before `b` if `a + b > b + a` as strings.

In [10]:
from functools import cmp_to_key

def largest_number(nums):
    strs = [str(x) for x in nums]
    def compare(a, b):                       # put a first if a+b is bigger
        if a + b > b + a:
            return -1
        if a + b < b + a:
            return 1
        return 0
    strs.sort(key=cmp_to_key(compare))
    result = "".join(strs)
    return "0" if result[0] == "0" else result

assert largest_number([10, 2]) == "210"
assert largest_number([3, 30, 34, 5, 9]) == "9534330"
assert largest_number([0, 0]) == "0"
print("Largest Number ")

Largest Number 


### 5. Count of Inversions (merge-sort trick) - classic interview question
Count pairs `i < j` with `a[i] > a[j]` in O(n log n). (Measures how 'unsorted' rankings are - used in recommender systems to compare rankings.)

**Idea:** During merge, when an element from the right half is taken first, it forms an inversion with every remaining element in the left half.

In [11]:
def count_inversions(a):
    def sort_count(arr):
        if len(arr) <= 1:
            return arr, 0
        mid = len(arr) // 2
        left, inv_l = sort_count(arr[:mid])
        right, inv_r = sort_count(arr[mid:])
        merged, i, j, inv = [], 0, 0, inv_l + inv_r
        while i < len(left) and j < len(right):
            if left[i] <= right[j]:
                merged.append(left[i]); i += 1
            else:
                merged.append(right[j]); j += 1
                inv += len(left) - i          # right[j] is smaller than all remaining left items
        merged += left[i:] + right[j:]
        return merged, inv
    return sort_count(a)[1]

assert count_inversions([2, 4, 1, 3, 5]) == 3        # (2,1), (4,1), (4,3)
assert count_inversions([5, 4, 3, 2, 1]) == 10
assert count_inversions([1, 2, 3]) == 0
print("Count Inversions ")

Count Inversions 


## More practice on LeetCode
Easy: Merge Sorted Array (#88), Majority Element (#169), Height Checker (#1051)
Medium: Kth Largest Element (#215 - quickselect), Sort List (#148 - merge sort on a linked list), H-Index (#274), Wiggle Sort (#280)
Hard: Count of Smaller Numbers After Self (#315), Reverse Pairs (#493)

## Key takeaways
- Know merge sort and quicksort cold - code + complexity + stability.
- In real code use `sorted(..., key=...)` (Timsort: stable, O(n log n)).
- "Sort first" unlocks two pointers, greedy and interval solutions.

**Next: `02.4-Graph-Algorithms.ipynb`**